![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Disponibilización de modelos

En este notebook aprenderá a guardar un modelo y a disponibilizarlo como una API con la librería Flask. Una API (interfaz de programación de aplicaciones) es un conjunto de definiciones y protocolos que permiten que servicios, en este caso modelos, retornen resultados y respuestas sin necesidad de saber cómo están implementados.

## Instrucciones Generales:

Este notebook esta compuesto por dos secciones. En la primera secciónn, usted beberá entrenar y guardar (exportar) un modelo de random forest para predecir si una URL es phishing (fraudulenta) o no. En la segunda parte, usará el modelo entrenado y lo disponibilizara usando la libreria *Flask*. En el siguente paper puede conocer más detalles de la base de datos que usaremos y del problema: *A. Correa Bahnsen, E. C. Bohorquez, S. Villegas, J. Vargas, and F. A. Gonzalez, “Classifying phishing urls using recurrent neural networks,” in Electronic Crime Research (eCrime), 2017 APWG Symposium on. IEEE, 2017, pp. 1–8*. https://albahnsen.files.wordpress.com/2018/05/classifying-phishing-urls-using-recurrent-neural-networks_cameraready.pdf
  
Para realizar la actividad, solo siga las indicaciones asociadas a cada celda del notebook. 


## Importar base de datos y librerías

In [1]:
#!pip install -r requirements.txt

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Importación librerías
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import joblib

import os
os.chdir('..')

In [4]:
# Carga de datos de archivos .csv
data = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/phishing.csv')
data.head()

,url,phishing
0,http://www.subalipack.com/contact/images/sampl...,1
1,http://fasc.maximecapellot-gypsyjazz-ensemble....,1
2,http://theotheragency.com/confirmer/confirmer-...,1
3,http://aaalandscaping.com/components/com_smart...,1
4,http://paypal.com.confirm-key-21107316126168.s...,1


## Codificar variables categóricas
Relizar preprocesamiento de texto (URLs) para crear variables predictoras:

In [5]:
# Creación de columnas binarias que indican si la URL contiene la palabra clave (keywords)
keywords = ['https', 'login', '.php', '.html', '@', 'sign']
for keyword in keywords:
    data['keyword_' + keyword] = data.url.str.contains(keyword).astype(int)

# Definición de la variable largo de la URL
data['lenght'] = data.url.str.len() - 2

# Definición de la variable largo del dominio de la URL
domain = data.url.str.split('/', expand=True).iloc[:, 2]
data['lenght_domain'] = domain.str.len()

# Definición de la variable binaria que indica si es IP
data['isIP'] = (domain.str.replace('.', '') * 1).str.isnumeric().astype(int)

# Definicón de la variable cuenta de 'com' en la URL
data['count_com'] = data.url.str.count('com')

data.head()

,url,phishing,keyword_https,keyword_login,keyword_.php,keyword_.html,keyword_@,keyword_sign,lenght,lenght_domain,isIP,count_com
0,http://www.subalipack.com/contact/images/sampl...,1,0,0,0,0,0,0,47,18,0,1
1,http://fasc.maximecapellot-gypsyjazz-ensemble....,1,0,0,0,0,0,0,73,41,0,0
2,http://theotheragency.com/confirmer/confirmer-...,1,0,0,0,0,0,0,92,18,0,1
3,http://aaalandscaping.com/components/com_smart...,1,0,0,0,0,0,0,172,18,0,3
4,http://paypal.com.confirm-key-21107316126168.s...,1,0,0,0,0,0,0,90,50,0,1


In [6]:
# Separación de variables predictoras (X) y variable de interes (y)
X = data.drop(['url', 'phishing'], axis=1)
y = data.phishing

## Entrenar y guardar el modelo

In [7]:
# Definición de modelo de clasificación Random Forest
clf = RandomForestClassifier(n_jobs=-1, n_estimators=100, max_depth=3)
cross_val_score(clf, X, y, cv=10)

array([0.75225, 0.75325, 0.7445 , 0.75325, 0.74775, 0.7605 , 0.7565 ,
       0.75625, 0.751  , 0.75625])

In [8]:
# Entrenamiento del modelo de clasificación Random Forest
clf.fit(X, y)

RandomForestClassifier(max_depth=3, n_jobs=-1)

In [9]:
# Exportar modelo a archivo binario .pkl
joblib.dump(clf, 'model_deployment/phishing_clf.pkl', compress=3)

['model_deployment/phishing_clf.pkl']

In [10]:
# Importar modelo y predicción
from model_deployment.m09_model_deployment import predict_proba

# Predicción de probabilidad de que un link sea phishing
predict_proba('http://www.vipturismolondres.com/com.br/?atendimento=Cliente&/LgSgkszm64/B8aNzHa8Aj.php')

0.7318924106369917

## Disponibilizar modelo con Flask

Para esta sección del notebook instale las siguientes librerías *!pip install flask* y *!pip install flask_restplus*.

In [12]:
# Importación librerías
from flask import Flask
from flask_restx import Api, Resource, fields

In [14]:
# Se importa la clase Flask para crear la aplicación web
app = Flask(__name__)

# Se define la API de Flask-RESTPlus, especificando:
# - app: la aplicación Flask que usará esta API
# - version: la versión de la API
# - title: el título que tendrá la documentación de la API
# - description: una breve descripción de lo que hace la API
api = Api(
    app, 
    version='1.0', 
    title='Phishing Prediction API',
    description='Phishing Prediction API'
)

# Se crea un "namespace" (espacio de nombres) para organizar los endpoints relacionados con predicciones
# En este caso, los endpoints estarán bajo la ruta '/predict'
ns = api.namespace('predict', 
     description='Phishing Classifier')

# Se definen los argumentos/parámetros que el endpoint aceptará como entrada
# En este caso, se espera un parámetro llamado 'URL', que debe ser un string, es obligatorio,
# y se encuentra en los argumentos de la URL (query string)
parser = ns.parser()
parser.add_argument(
    'URL', 
    type=str, 
    required=True, 
    help='URL to be analyzed', 
    location='args'  # significa que el parámetro se espera en la URL (por ejemplo: ?URL=http://...)
)

# Se define el modelo de salida esperado en las respuestas de la API
# Este modelo indica que la respuesta será un objeto con un campo 'result' de tipo string
resource_fields = api.model('Resource', {
    'result': fields.String,
})


In [15]:
# Se define la clase que representa el recurso disponible en el endpoint '/predict/'
# Esta clase hereda de `Resource` de Flask-RESTPlus, lo que permite manejar solicitudes HTTP como GET, POST, etc.
@ns.route('/')
class PhishingApi(Resource):

    # Se documenta que este método GET utiliza los argumentos definidos previamente con el parser
    @ns.doc(parser=parser)

    # Se especifica el formato de la respuesta, basado en el modelo 'resource_fields'
    @ns.marshal_with(resource_fields)
    def get(self):
        # Se obtienen los argumentos enviados en la solicitud (en este caso, solo la URL)
        args = parser.parse_args()
        
        # Se retorna un diccionario con el resultado de la predicción, utilizando la función predict_proba
        # Se asume que predict_proba es una función previamente definida que toma una URL y devuelve una predicción
        return {
         "result": predict_proba(args['URL'])
        }, 200  # Se devuelve junto con un código HTTP 200 que indica éxito


In [ ]:
# Ejecuta la aplicación Flask localmente
# Parámetros:
# - debug=True: permite mostrar mensajes de depuración detallados (útil durante el desarrollo)
# - use_reloader=False: evita que Flask reinicie automáticamente la aplicación al detectar cambios (opcional)
# - host='0.0.0.0': expone la aplicación en todas las interfaces de red (útil si se quiere acceder desde otra máquina)
# - port=5000: el puerto en el que se ejecutará la API (por defecto, Flask usa el 5000)
app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.224.165:5000
Press CTRL+C to quit
192.168.224.165 - - [11/Apr/2025 11:27:03] "GET / HTTP/1.1" 200 -
192.168.224.165 - - [11/Apr/2025 11:27:04] "GET /swaggerui/swagger-ui.css HTTP/1.1" 200 -
192.168.224.165 - - [11/Apr/2025 11:27:04] "GET /swaggerui/swagger-ui-standalone-preset.js HTTP/1.1" 200 -
192.168.224.165 - - [11/Apr/2025 11:27:04] "GET /swaggerui/droid-sans.css HTTP/1.1" 200 -
192.168.224.165 - - [11/Apr/2025 11:27:04] "GET /swaggerui/swagger-ui-bundle.js HTTP/1.1" 200 -
192.168.224.165 - - [11/Apr/2025 11:27:04] "GET /swaggerui/favicon-32x32.png HTTP/1.1" 200 -
192.168.224.165 - - [11/Apr/2025 11:27:04] "GET /swagger.json HTTP/1.1" 200 -
192.168.224.165 - - [11/Apr/2025 11:27:50] "GET /predict/?URL=.com.com.com.com HTTP/1.1" 500 -
Traceback (most recent call last):
  File "c:\Users\sjaramillo\AppData\Local\anaconda3\envs\General\Lib\site-packages\flask\app.py", line 1536, in _

El modelo debe haber quedado disponibilizado en el puerto 5000. Para predecir la probabilidad de que una URL sea fraudulenta (phishing) copie en la barra de busqueda de su navegador la siguiente dirección (http://localhost:5000/predict/?URL=) y agregregue al final de esta la URL que desee precir. Por ejemplo, al copiar la URL http://localhost:5000/predict/?URL=http://consultoriojuridico.co/pp/www.paypal.com/, la API retornará la probabilidad de que la URL http://consultoriojuridico.co/pp/www.paypal.com/ sea phishing.